# Transferencia del aprendizaje (Transfer Learning) en PyTorch

En la actualidad, los modelos con mejor rendimiento en el campo del aprendizaje automático son las redes neuronales con una gran cantidad de parámetros, como se puede ver en algunos de los modelos disponibles en `torchvision` ([torchvision.models](https://pytorch.org/vision/stable/models.html)). Por ejemplo, se estima que el modelo ChatGPT cuenta con más de 175 mil millones de parámetros. Este tipo de modelos requieren grandes cantidades de datos y recursos computacionales para su entrenamiento, lo que puede resultar difícil de alcanzar para algunos. Se estima que Microsoft invirtió alrededor de 3 mil millones de dólares en la construcción del modelo ChatGPT (fuente: [New York Times](https://www.nytimes.com/2023/01/12/technology/microsoft-openai-chatgpt.html)).

Este hecho presenta un problema en el contexto de los científicos de datos, ya que a menudo no se cuenta con acceso a grandes cantidades de datos ni a la infraestructura computacional necesaria para entrenar modelos profundos. Una alternativa es la **transferencia del aprendizaje** (*Transfer Learning*): partir de un modelo que otros ya entrenaron y adaptarlo a nuestro problema.

Lo haremos con un problema de clasificación binaria: distinguir imágenes de **gatos** y **perros** con la base de datos [Cats vs Dogs](https://www.tensorflow.org/datasets/catalog/cats_vs_dogs?hl=es-419).

**Al terminar este notebook podrás:**

- Cargar un modelo pre-entrenado (VGG16) y quedarte con su parte convolucional.
- Congelar sus parámetros y agregar tus propias capas finales.
- Contar cuántos parámetros se entrenan realmente y entender por qué son pocos.
- Organizar todo el flujo de entrenamiento en una clase.

La intuición es la siguiente: una red convolucional entrenada con millones de imágenes aprende, en sus primeras capas, a detectar bordes y texturas, y en las capas más profundas, partes de objetos. Todo eso también sirve para distinguir gatos de perros, así que no tenemos que aprenderlo de nuevo.

En este notebook usaremos **PyTorch**. A diferencia de otras librerías que ocultan el ciclo de entrenamiento dentro de una función (`fit`), en PyTorch lo escribimos nosotros mismos. Esto requiere unas pocas líneas más, pero nos permite ver exactamente qué ocurre en cada paso del aprendizaje.

## Etapa 1: Carga de los datos

Se cargan las librerías necesarias. Si hay una GPU disponible, la usaremos para acelerar el entrenamiento; de lo contrario, se usa la CPU. Tanto el modelo como los datos deben estar en el mismo dispositivo. Entrenar una red como VGG16 en CPU es lento; si estás en Google Colab, activa una GPU en *Entorno de ejecución > Cambiar tipo de entorno de ejecución*. Fijamos también una semilla (`manual_seed`) para que los resultados sean reproducibles.

In [ ]:
import os
import urllib.request
import zipfile

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import datasets, models, transforms
from PIL import Image
import matplotlib.pyplot as plt
from tqdm import tqdm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.manual_seed(0)
print('Dispositivo:', device)

La base de datos Cats vs Dogs **no** viene incluida en `torchvision`, así que la descargamos (unos 790 MB) desde Microsoft y la descomprimimos. El resultado es una carpeta con una subcarpeta por clase (`Cat` y `Dog`). Esta es una organización muy común para bases de datos de imágenes, y `torchvision` ofrece `datasets.ImageFolder` para leerla: usa el nombre de cada subcarpeta como etiqueta.

In [ ]:
data_dir = '../../../data'
url = 'https://download.microsoft.com/download/3/E/1/3E1C3F21-ECDB-4869-8368-6DEBA77B919F/kagglecatsanddogs_5340.zip'
zip_path = os.path.join(data_dir, 'cats_and_dogs.zip')
images_dir = os.path.join(data_dir, 'PetImages')

os.makedirs(data_dir, exist_ok=True)
if not os.path.isdir(images_dir):
    print('Descargando...')
    urllib.request.urlretrieve(url, zip_path)
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(data_dir)
    os.remove(zip_path)
print(os.listdir(images_dir))

Esta base de datos contiene algunas imágenes dañadas (archivos vacíos o corruptos). Si las dejamos, el entrenamiento se detendría con un error en medio de una época. Por eso, le pasamos a `ImageFolder` una función que intenta abrir cada imagen y descarta las que fallan. Este paso tarda un par de minutos, pero solo se hace una vez.

In [ ]:
def is_valid_image(path):
    if not path.lower().endswith('.jpg'):
        return False
    try:
        with Image.open(path) as img:
            img.load()  # fuerza la lectura completa de la imagen
        return True
    except Exception:
        return False

# transform=None: por ahora cada elemento es una imagen PIL; las transformaciones se agregan después.
full_data = datasets.ImageFolder(images_dir, transform=None, is_valid_file=is_valid_image)
class_names = full_data.classes
print('Clases:', class_names)
print('Imágenes válidas:', len(full_data))

Separamos tres conjuntos, igual que en el notebook de clasificación de dígitos:

| Conjunto | Proporción | Para qué sirve |
|---|---|---|
| Entrenamiento | 70 % | Calcular los pesos de la red |
| Validación | 15 % | Vigilar el desempeño época a época |
| Prueba (test) | 15 % | Evaluación final, una sola vez, al terminar |

Como la base de datos viene en un solo bloque, la dividimos nosotros con `random_split`. Fijamos una semilla para que la división sea siempre la misma.

In [ ]:
generator = torch.Generator().manual_seed(42)
n_total = len(full_data)
n_train = int(0.70 * n_total)
n_val = int(0.15 * n_total)
n_test = n_total - n_train - n_val
train_subset, val_subset, test_subset = random_split(full_data, [n_train, n_val, n_test], generator=generator)
print('Entrenamiento:', len(train_subset), '| Validación:', len(val_subset), '| Prueba:', len(test_subset))

Veamos algunos ejemplos del conjunto de prueba:

In [ ]:
fig, axes = plt.subplots(1, 8, figsize=(16, 3))
for ax, idx in zip(axes, test_subset.indices[:8]):
    image, label = full_data[idx]
    ax.axis('off')
    ax.set_title(class_names[label])
    ax.imshow(image)
plt.show()

## Etapa 2: Procesamiento de los datos

Las imágenes tienen tamaños distintos, y una red necesita que todas las entradas tengan el mismo tamaño. Por eso las redimensionamos a $150 \times 150$ píxeles ($H = W = 150$).

Además, hacemos dos conversiones:

1. `ToTensor()` convierte la imagen en un tensor y normaliza los píxeles del rango $[0, 255]$ al intervalo $[0, 1]$. También reordena las dimensiones a `(canales, H, W)`.
2. `Normalize(mean, std)` resta a cada canal RGB una media y divide por una desviación estándar. **Este paso es obligatorio al usar un modelo pre-entrenado:** VGG16 fue entrenada con imágenes normalizadas con las estadísticas de ImageNet (los valores de abajo), así que nuestras imágenes deben recibir el mismo tratamiento para que la red las "entienda".

In [ ]:
imagenet_mean = [0.485, 0.456, 0.406]
imagenet_std = [0.229, 0.224, 0.225]

preprocess = transforms.Compose([
    transforms.Resize((150, 150)),
    transforms.ToTensor(),
    transforms.Normalize(imagenet_mean, imagenet_std),
])

`random_split` nos entrega subconjuntos que comparten una misma base de datos, y esa base no tiene transformaciones. Para poder aplicar una transformación distinta a cada conjunto, definimos una pequeña clase `TransformedSubset`. Una clase `Dataset` de PyTorch solo necesita dos métodos: `__len__` (cuántos datos hay) y `__getitem__` (cómo obtener el dato número `i`).

In [ ]:
class TransformedSubset(Dataset):
    def __init__(self, subset, transform):
        self.subset = subset
        self.transform = transform

    def __len__(self):
        return len(self.subset)

    def __getitem__(self, i):
        image, label = self.subset[i]
        return self.transform(image), label

Ahora creamos los `DataLoader`, que entregan los datos en **minibatches** de tamaño 32. En el conjunto de entrenamiento mezclamos (`shuffle=True`) los datos en cada época. `num_workers` indica cuántos procesos leen y transforman las imágenes en paralelo.

In [ ]:
batch_size = 32
train_loader = DataLoader(TransformedSubset(train_subset, preprocess), batch_size=batch_size,
                          shuffle=True, num_workers=2)
val_loader = DataLoader(TransformedSubset(val_subset, preprocess), batch_size=batch_size,
                        shuffle=False, num_workers=2)
test_loader = DataLoader(TransformedSubset(test_subset, preprocess), batch_size=batch_size,
                         shuffle=False, num_workers=2)

images, labels = next(iter(train_loader))
print(images.shape)  # (batch, canales, H, W)
print(labels.shape)

## Etapa 3: Definición del modelo por transferencia del aprendizaje

La técnica de *transfer learning* consiste en aprovechar un modelo previamente entrenado y adaptarlo a nuestro problema de aprendizaje específico. De esta forma, se requiere una cantidad menor de datos para entrenar el modelo, lo que reduce el riesgo de sobreajuste y disminuye los requisitos de recursos computacionales.

### 3.1 El modelo base

Cargamos una arquitectura **VGG16**, entrenada con la base de datos ImageNet (más de un millón de imágenes en 1000 clases). Un modelo de `torchvision` se divide en dos partes:

- `features`: las capas convolucionales, que extraen las características de la imagen.
- `classifier`: las capas densas finales, que están ligadas a la tarea original (clasificar las 1000 clases de ImageNet).

Nosotros solo conservamos `features` y descartamos `classifier`, pues diseñaremos nuestras propias capas finales para la tarea de gatos y perros. La primera vez, `torchvision` descarga los pesos (unos 530 MB).

In [ ]:
vgg = models.vgg16(weights=models.VGG16_Weights.IMAGENET1K_V1)
base_model = vgg.features
print(base_model)

In [ ]:
# Esta red tiene más de 14 millones de parámetros.
base_params = sum(p.numel() for p in base_model.parameters())
print('Parámetros del modelo base:', base_params)

### 3.2 Congelar el modelo base

Como hemos dicho, nosotros **no vamos a entrenar** los parámetros del modelo base; solo los de las capas que diseñemos. Para ello, le indicamos a PyTorch que no calcule gradientes para esos parámetros con `requires_grad = False`. Claramente esto reduce la cantidad necesaria de recursos computacionales y de datos. Decimos que el modelo base está **congelado**.

In [ ]:
for param in base_model.parameters():
    param.requires_grad = False

### 3.3 Las capas nuevas

Ahora diseñamos las capas superiores de la red, para que sean las adecuadas para la tarea planteada. Para saber cuántas entradas debe tener la primera capa densa, pasamos una imagen de ejemplo por el modelo base y miramos la forma de la salida:

In [ ]:
with torch.no_grad():
    features = base_model(images[:1])
print(features.shape)  # (batch, canales, H, W)
print('Unidades tras Flatten:', features.numel())

La salida es de tamaño $512 \times 4 \times 4$: 512 mapas de características de $4 \times 4$. Al aplastarlos (`Flatten`) obtenemos $512 \cdot 4 \cdot 4 = 8192$ unidades, que alimentan las capas densas.

Armamos la red completa con `nn.Sequential`: el modelo base seguido de nuestras capas. **Nota:** la última capa no aplica `sigmoid`. La red entrega un *logit* y la función de pérdida `nn.BCEWithLogitsLoss` aplica internamente la sigmoide.

In [ ]:
net = nn.Sequential(
    base_model,  # se parte del modelo base.

    # Se agregan algunas capas para la tarea de aprendizaje. En este punto no se
    # suele usar capas convolucionales, solo capas densas.
    nn.Flatten(),  # 512 x 4 x 4 = 8192 unidades
    nn.Linear(512 * 4 * 4, 256),
    nn.ReLU(),
    nn.Dropout(0.5),

    # La última capa corresponde a la tarea: clasificación de 2 clases. Usamos una
    # sola unidad cuya salida indica qué tan probable es que la imagen sea un perro.
    nn.Linear(256, 1),
).to(device)

Contemos cuántos parámetros tiene el modelo y cuántos se van a entrenar:

In [ ]:
total_params = sum(p.numel() for p in net.parameters())
trainable_params = sum(p.numel() for p in net.parameters() if p.requires_grad)
print('Parámetros totales:     ', f'{total_params:,}')
print('Parámetros entrenables: ', f'{trainable_params:,}')
print('Parámetros congelados:  ', f'{total_params - trainable_params:,}')

De los más de 16 millones de parámetros que tiene esta red, únicamente serán entrenados cerca de 2 millones (las dos capas densas que agregamos). Esto significa una reducción importante en la cantidad de datos y, por consiguiente, en los recursos computacionales.

### 3.4 Función de pérdida, optimizador y métrica

- **`BCEWithLogitsLoss`**: entropía cruzada binaria para clasificación de 2 clases (incluye la sigmoide). Espera etiquetas decimales (`float`) de forma `(batch,)`, y nuestra red entrega `(batch, 1)`, por eso usaremos `squeeze(1)`.
- **`Adam`**: el optimizador ADAM. **Solo le pasamos los parámetros que tienen `requires_grad=True`**, es decir, los de las capas nuevas.
- **Accuracy**: un logit mayor que 0 equivale a una probabilidad mayor que 0.5, es decir, a predecir "perro".

In [ ]:
loss_fn = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam([p for p in net.parameters() if p.requires_grad], lr=1e-3)

def accuracy(logits, y):
    # Un logit mayor que 0 equivale a una probabilidad mayor que 0.5: se predice "perro" (clase 1)
    return ((logits > 0).float() == y).float().mean().item()

### 3.5 Una iteración de entrenamiento

Antes de entrenar durante muchas épocas, veamos una sola **iteración**: un minibatch que entra a la red y actualiza los pesos. Los cinco pasos son los mismos del notebook de clasificación de dígitos:

1. **`optimizer.zero_grad()`**: borra los gradientes de la iteración anterior.
2. **Forward, `net(x)`**: la red calcula su predicción.
3. **Pérdida, `loss_fn(logits, y)`**: mide qué tan lejos está la predicción de la etiqueta real.
4. **`loss.backward()`**: calcula el gradiente de la pérdida respecto a cada parámetro **entrenable**. Los parámetros congelados no reciben gradiente.
5. **`optimizer.step()`**: actualiza los pesos.

In [ ]:
x, y = images.to(device), labels.to(device).float()

net.train()
optimizer.zero_grad()            # 1. limpiar los gradientes
logits = net(x).squeeze(1)       # 2. forward
loss = loss_fn(logits, y)        # 3. error entre predicción y etiqueta
loss.backward()                  # 4. backward
optimizer.step()                 # 5. actualizar los pesos
print(f'Pérdida del minibatch: {loss.item():.4f} | accuracy: {accuracy(logits, y):.4f}')

Podemos comprobar que el modelo base realmente está congelado: sus parámetros no tienen gradiente, mientras que los de las capas nuevas sí.

In [ ]:
print('Gradiente en la primera convolución:', base_model[0].weight.grad)
print('Gradiente en la primera capa densa: ', net[2].weight.grad is not None)

## Etapa 4: Organizar el código en una clase

Todo lo que hicimos hasta aquí funciona, pero está en celdas sueltas. Cuando queremos probar otra configuración o repetir el entrenamiento, hay que copiar y pegar mucho código.

Una forma más ordenada es agrupar todo en una **clase** que tenga un método por cada tarea:

| Método | Qué hace |
|---|---|
| `__init__` | Carga el modelo base, lo congela, agrega las capas nuevas y crea la pérdida y el optimizador. |
| `train_epoch` | Recorre los minibatches de una época y actualiza los pesos. |
| `evaluate` | Mide pérdida y accuracy sobre un conjunto, sin entrenar. |
| `fit` | Repite `train_epoch` y `evaluate` por varias épocas y guarda el historial. |
| `predict` | Devuelve la clase predicha para unas imágenes. |

Es exactamente el mismo código de las secciones anteriores, solo reorganizado. La clase usa `device` y `accuracy`, que definimos antes en el notebook. Además, los tamaños de las capas nuevas y la tasa de aprendizaje quedan como argumentos.

In [ ]:
class CatsDogsClassifier:
    def __init__(self, hidden_units=256, dropout=0.5, lr=1e-3):
        # Modelo base: la parte convolucional de VGG16 pre-entrenada en ImageNet, congelada
        base_model = models.vgg16(weights=models.VGG16_Weights.IMAGENET1K_V1).features
        for param in base_model.parameters():
            param.requires_grad = False

        # Capas nuevas: son las únicas que se entrenan
        self.net = nn.Sequential(
            base_model,
            nn.Flatten(),                         # 512 x 4 x 4 = 8192 unidades
            nn.Linear(512 * 4 * 4, hidden_units),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_units, 1),           # un solo logit: qué tan probable es "perro"
        ).to(device)

        self.loss_fn = nn.BCEWithLogitsLoss()
        # Al optimizador solo le pasamos los parámetros que no están congelados
        trainable = [p for p in self.net.parameters() if p.requires_grad]
        self.optimizer = torch.optim.Adam(trainable, lr=lr)
        self.history = {'loss': [], 'accuracy': [], 'val_loss': [], 'val_accuracy': []}

    def train_epoch(self, loader):
        # Una época: recorre todos los minibatches y actualiza los pesos
        self.net.train()
        total_loss, total_acc, n = 0.0, 0.0, 0
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device).float()
            self.optimizer.zero_grad()
            logits = self.net(xb).squeeze(1)
            loss = self.loss_fn(logits, yb)
            loss.backward()
            self.optimizer.step()
            total_loss += loss.item() * len(yb)
            total_acc += accuracy(logits, yb) * len(yb)
            n += len(yb)
        return total_loss / n, total_acc / n

    def evaluate(self, loader):
        # Mide el desempeño sin entrenar (sin calcular gradientes)
        self.net.eval()
        total_loss, total_acc, n = 0.0, 0.0, 0
        with torch.no_grad():
            for xb, yb in loader:
                xb, yb = xb.to(device), yb.to(device).float()
                logits = self.net(xb).squeeze(1)
                total_loss += self.loss_fn(logits, yb).item() * len(yb)
                total_acc += accuracy(logits, yb) * len(yb)
                n += len(yb)
        return total_loss / n, total_acc / n

    def fit(self, train_loader, val_loader, epochs=20):
        pbar = tqdm(range(epochs), desc='Entrenamiento', unit='época')
        for epoch in pbar:
            loss, acc = self.train_epoch(train_loader)
            val_loss, val_acc = self.evaluate(val_loader)
            for key, value in zip(self.history, (loss, acc, val_loss, val_acc)):
                self.history[key].append(value)
            pbar.set_postfix(loss=f'{loss:.4f}', accuracy=f'{acc:.4f}',
                             val_loss=f'{val_loss:.4f}', val_accuracy=f'{val_acc:.4f}')

    def predict(self, x):
        # Devuelve la clase predicha (0 = gato, 1 = perro) para cada imagen del minibatch
        self.net.eval()
        with torch.no_grad():
            return (self.net(x.to(device)).squeeze(1) > 0).long().cpu()

Con la clase definida, entrenar el modelo completo se reduce a dos líneas. Entrenamos con minibatches de tamaño 32 durante 20 épocas (en GPU tarda unos minutos; en CPU puede tardar bastante más, así que puedes reducir `epochs` para probar).

Una nota sobre `train()`: la parte `features` de VGG16 no tiene capas Dropout ni BatchNorm, por lo que ponerla en modo entrenamiento no cambia su comportamiento y los pesos congelados se mantienen intactos.

In [ ]:
clf = CatsDogsClassifier(hidden_units=256, dropout=0.5, lr=1e-3)
clf.fit(train_loader, val_loader, epochs=20)

## Etapa 5: Evaluación

Graficamos la evolución de la pérdida y del accuracy. Qué buscar:

- La pérdida **baja** y el accuracy **sube** época tras época: la red está aprendiendo.
- Si la curva de entrenamiento sigue mejorando pero la de validación se estanca o empeora, la red está memorizando los datos de entrenamiento (**sobreajuste**).

In [ ]:
def plot_history(history):
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))

    axes[0].plot(history['accuracy'], label='entrenamiento')
    axes[0].plot(history['val_accuracy'], label='validación')
    axes[0].set_title('Accuracy del modelo')
    axes[0].set_xlabel('época')
    axes[0].set_ylabel('accuracy')
    axes[0].legend()

    axes[1].plot(history['loss'], label='entrenamiento')
    axes[1].plot(history['val_loss'], label='validación')
    axes[1].set_title('Pérdida del modelo')
    axes[1].set_xlabel('época')
    axes[1].set_ylabel('pérdida')
    axes[1].legend()

    plt.tight_layout()
    plt.show()

plot_history(clf.history)

**Qué vemos en estas curvas:** el accuracy de validación es alto desde las primeras épocas: el modelo base ya "sabía" extraer características útiles. Pero la pérdida de entrenamiento sigue bajando mientras la de validación se estanca o sube. Eso es **sobreajuste**: aunque redujimos los parámetros entrenables de 16 a 2 millones, siguen siendo muchos para la cantidad de imágenes de entrenamiento disponibles.

En el siguiente notebook veremos cómo atacar este problema con el **aumento de datos**.

Ahora usamos el conjunto de **prueba**, que la red no vio ni durante el entrenamiento ni durante la validación. Este número es la mejor estimación de qué tan bien funcionará con imágenes nuevas.

In [ ]:
test_loss, test_acc = clf.evaluate(test_loader)
print(f'Pérdida en prueba: {test_loss:.4f}')
print(f'Accuracy en prueba: {test_acc:.4f}')

### Ejemplos de predicciones

Veamos algunas imágenes de prueba que la red clasificó mal. Para mostrarlas, deshacemos la normalización (`imagenet_mean` y `imagenet_std`) y recuperamos los colores originales.

In [ ]:
def denormalize(image):
    mean = torch.tensor(imagenet_mean).view(3, 1, 1)
    std = torch.tensor(imagenet_std).view(3, 1, 1)
    return (image * std + mean).clamp(0, 1).permute(1, 2, 0)  # (C, H, W) -> (H, W, C)

wrong_images, wrong_true, wrong_pred = [], [], []
for xb, yb in test_loader:
    preds = clf.predict(xb)
    mask = preds != yb
    for img, true, pred in zip(xb[mask], yb[mask], preds[mask]):
        wrong_images.append(img); wrong_true.append(true.item()); wrong_pred.append(pred.item())
    if len(wrong_images) >= 12:
        break

plt.figure(figsize=(10, 6))
for k in range(min(12, len(wrong_images))):
    plt.subplot(3, 4, k + 1)
    plt.imshow(denormalize(wrong_images[k]))
    plt.title(f'Pred: {class_names[wrong_pred[k]]} | Real: {class_names[wrong_true[k]]}', color='red', fontsize=9)
    plt.axis('off')
plt.tight_layout()
plt.show()

## Resumen

- **Transfer learning** consiste en reutilizar un modelo pre-entrenado y adaptarlo a una tarea nueva, lo que reduce la cantidad de datos y de cómputo necesarios.
- En `torchvision` cargamos el modelo con `models.vgg16(weights=...)` y nos quedamos con `.features`, su parte convolucional.
- Se **congela** el modelo base con `requires_grad = False` y se agregan capas nuevas (aplastar, densas, Dropout, salida).
- Al usar un modelo pre-entrenado, las imágenes deben **normalizarse con las mismas estadísticas** con que se entrenó.
- Para clasificación binaria usamos una sola salida (logit) con `BCEWithLogitsLoss`.
- Aun con pocos parámetros entrenables aparece **sobreajuste**; el aumento de datos es una forma de reducirlo.
- Todo el flujo se agrupa en una **clase** con métodos como `fit`, `evaluate` y `predict`.

## Ejercicios

Modifica un valor a la vez, vuelve a crear `clf` y entrena. Observa cómo cambian las curvas.

1. Cambia `lr` a `1e-4` y luego a `1e-2`. ¿Qué pasa con la velocidad de aprendizaje y con la estabilidad?
2. Cambia `hidden_units` a `64` y luego a `1024`. ¿Cuántos parámetros entrenables hay en cada caso? ¿Cambia el sobreajuste?
3. Cambia `dropout` a `0.0` y luego a `0.8`. ¿Qué efecto tiene sobre la brecha entre entrenamiento y validación?
4. Reemplaza VGG16 por otro modelo de `torchvision`, por ejemplo `models.resnet18`. (Pista: revisa la forma de salida de su parte convolucional para ajustar la primera capa densa.)
5. **Para pensar:** ¿qué pasaría si NO congeláramos el modelo base (`requires_grad = True`)? ¿Cuántos parámetros se entrenarían y qué esperarías del tiempo de entrenamiento?